# Pokemon Type Classification — DL Assignment 1

**Goal:** Classify Pokemon images into 9 types using MLP → CNN → Transfer Learning.  
**Metric:** Macro-averaged F1 (chosen because of class imbalance — 2.76x ratio between majority and minority class).  
**Data:** 3600 train / 900 test, 64×64 PNG images, 9 classes.  
**Tasks:** Task 1 — MLP baseline | Task 2 — Custom CNN | Task 3 — EfficientNet Transfer Learning.

---
*Run this notebook top to bottom. Each section calls the tested `src/` Python files and displays results inline.*  
*All logic lives in `src/` — this notebook is a runner + display layer only.*

## How to run this notebook

**Locally** — just run all cells. `src/` and `data/` are already on disk.

**Colab** — run cells 1 and 2 once per session:
- **Cell 1 (Setup):** clones the repo, installs dependencies.
- **Cell 2 (Data):** downloads the dataset from Google Drive automatically.

After those two cells, everything else runs identically locally and on Colab.


In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
# Locally:  notebook lives in task1/ — we step up to assignment_1/ so all
#           relative paths (src/, data/, outputs/) resolve correctly.
# Colab:    clones the repo, moves into assignment_1/, installs deps.
# Run once per session.
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/DL_Proj"):
        !git clone https://github.com/fmssilva/DL_Proj.git /content/DL_Proj
    else:
        !git -C /content/DL_Proj pull --ff-only
    os.chdir("/content/DL_Proj/assignment_1")
    %pip install -r requirements.txt -q
else:
    # locally the notebook kernel starts in task1/ — go up one level to assignment_1/
    cwd = Path(os.getcwd())
    if cwd.name.startswith("task"):
        os.chdir(cwd.parent)

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# ── Shared imports ────────────────────────────────────────────────────────────
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from src.config import (BATCH_SIZE, CLASSES, DATA_DIR, EPOCHS, IMG_SIZE_SMALL,
                        LR, NUM_WORKERS, OUT_DIR, PATIENCE, SEED,
                        create_output_dirs, set_seed)

set_seed(SEED)
create_output_dirs()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")

CSV_PATH  = DATA_DIR / "train_labels.csv"
TRAIN_DIR = DATA_DIR / "Train"
TEST_DIR  = DATA_DIR / "Test"
print("Ready:", ROOT)


Device: cpu  |  PyTorch: 2.5.1
Ready: c:\Users\franc\OneDrive\Nossa_Pasta_2\5. Universidade\Cadeiras\DL\DL_Proj\assignment_1\task1


In [4]:
# ── Cell 2: Data ─────────────────────────────────────────────────────────────
# Locally:  data/ already on disk — nothing to do.
# Colab:    downloads from Google Drive (no Kaggle account needed).
import os, zipfile
from pathlib import Path

if not Path("data/train_labels.csv").exists():
    if IN_COLAB:
        %pip install gdown -q
        import gdown
        gdown.download(id="1nVSQZxQubLEPXjSRqGn7rtPzkw-S0zIi", output="data.zip", quiet=False)
        with zipfile.ZipFile("data.zip") as zf:
            zf.extractall("data")
        os.remove("data.zip")
        print("data/ ready")
    else:
        print("ERROR: data/ not found — place data/ under assignment_1/")
else:
    print("data/ already present")

# ── Load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Loaded CSV: {len(df)} rows, {df['label'].nunique()} classes")


ERROR: data/ not found — place data/ under assignment_1/


FileNotFoundError: [Errno 2] No such file or directory: 'data\\train_labels.csv'

---
## Part 1 — Exploratory Data Analysis

Before training anything, we look at the data to understand:
- How many samples per class (imbalance ratio)
- Whether all images are the same size (justifies the resize choice)
- What the images actually look like per class
- Per-channel pixel statistics (justifies or challenges ImageNet normalisation)

All EDA runs on the full 3600-image training set.

In [ ]:
import src.datasets.eda as eda

print("=== Class Distribution ===")
eda.class_distribution(df)

print("\n=== Image Size Distribution ===")
eda.image_size_distribution(TRAIN_DIR)

print("\n=== Data Integrity Check ===")
valid, invalid = eda.check_data_integrity(TRAIN_DIR, df)
print(f"Result: {valid} valid, {invalid} invalid")


### Plot 1 — Class Distribution

Horizontal bar chart showing the number of samples per class, sorted from most to least frequent.

In [ ]:
import src.datasets.eda_plots as eda_plots

fig = eda_plots.plot_class_distribution(df)
plt.show()
plt.close(fig)


> **Finding:** _TODO — fill in after running: note the imbalance ratio, which classes are majority/minority, and what this means for the loss function choice._

### Plot 2 — Sample Images per Class

4 random images per class (fixed seed — same grid on every run).

In [ ]:
fig = eda_plots.plot_sample_images(TRAIN_DIR, df, n_per_class=4)
plt.show()
plt.close(fig)

> **Finding:** _TODO — note visually similar class pairs (e.g. Bug/Grass both greenish, Fighting/Normal both humanoid). These are the pairs most likely to be confused by the model._

### Plot 3 — Average Image per Class

Mean pixel value across all images in each class. A blurry mean image means high intra-class variance — the class has diverse-looking images, which makes it harder to classify.

In [ ]:
fig = eda_plots.plot_average_image_per_class(TRAIN_DIR, df)
plt.show()
plt.close(fig)

> **Finding:** _TODO — fill in after running_

### Plot 4 — Per-Channel Pixel Statistics

Bar chart of per-channel mean and standard deviation (R, G, B) computed across the training set.
Also prints numeric values to compare against ImageNet normalization constants.

In [ ]:
fig = eda_plots.plot_pixel_statistics(TRAIN_DIR, df)
plt.show()
plt.close(fig)

> **Finding:** _TODO — fill in after running_

### Plot 5 — Pixel Intensity Histogram

Histogram of pixel intensity values across R, G, B channels sampled from the training set.
Reveals the overall brightness distribution and confirms whether data is approximately normalized.

In [ ]:
fig = eda_plots.plot_pixel_intensity_histogram(TRAIN_DIR, df, n_samples=200)
plt.show()
plt.close(fig)

> **Finding:** _TODO — fill in after running_

---
## Task 1 — MLP Baseline

A fully-connected Multi-Layer Perceptron treating each 64×64 RGB image as a flat vector of 12,288 features.

**Architecture:** `Flatten → Linear(12288, 512) → BN → ReLU → Dropout(0.4) → Linear(512, 256) → BN → ReLU → Dropout(0.4) → Linear(256, 128) → BN → ReLU → Dropout(0.4) → Linear(128, 9)`

Total parameters: ~6.4M.

**Training strategy:**
- Loss: `CrossEntropyLoss` with class weights (handles 2.76× class imbalance)
- Optimizer: Adam (lr=1e-3), LR scheduler: StepLR (step_size=5, γ=0.5)
- Early stopping: patience=5 on validation loss, saves best checkpoint

**Tip:** set `FAST_RUN = True` in `src/config.py` to run 4 epochs for a quick pipeline check (~2 min CPU / ~1 min GPU). Set `False` for the full 30-epoch run on Colab.


In [ ]:
# ── Task 1 Step 1: Data loaders ───────────────────────────────────────────────
# augment=False, use_sampler=False for MLP — weighted loss is enough for 2.76x imbalance
import torch.nn as nn
from torch.utils.data import DataLoader
from src.datasets.dataset import (
    PokemonDataset, compute_class_weights,
    get_base_transforms, get_train_val_loaders,
)

train_loader, val_loader = get_train_val_loaders(
    CSV_PATH, TRAIN_DIR, IMG_SIZE_SMALL, BATCH_SIZE,
    augment=False, use_sampler=False, num_workers=NUM_WORKERS,
)
test_ds     = PokemonDataset(TEST_DIR, get_base_transforms(IMG_SIZE_SMALL), csv_path=None)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test images: {len(test_ds)}")

# ── Task 1 Step 2: Model, loss, optimiser, scheduler, early stopping ──────────
from src.models.mlp import MLP
from src.training.early_stopping import EarlyStopping

# compute class weights from the full CSV (not just one loader batch)
label_to_idx    = {cls: i for i, cls in enumerate(CLASSES)}
all_train_labels = [label_to_idx[lbl] for lbl in df["label"]]
class_weights   = compute_class_weights(all_train_labels).to(device)

CKPT_PATH = OUT_DIR / "checkpoints" / "task1_mlp_best.pth"

model     = MLP().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
stopper   = EarlyStopping(patience=PATIENCE, checkpoint_path=str(CKPT_PATH))

total_params = sum(p.numel() for p in model.parameters())
print(f"MLP params: {total_params:,}")


In [ ]:
# ── Task 1 Step 3: Training loop ──────────────────────────────────────────────
from src.training.train import train_one_epoch, evaluate

history = {"train_loss": [], "val_loss": [], "val_f1": []}

for epoch in range(1, EPOCHS + 1):
    t0         = time.time()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    elapsed    = time.time() - t0
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_f1"].append(val_metrics["macro_f1"])

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_f1={val_metrics['macro_f1']:.4f} | "
        f"time={elapsed:.1f}s"
    )

    stopper(val_metrics["loss"], model)
    if stopper.stop:
        print(f"Early stopping at epoch {epoch}")
        break

print("\nTraining done.")


In [ ]:
# ── Task 1 Step 4: Load best checkpoint + classification report ───────────────
from src.evaluation.metrics import classification_report_str

model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=True))
val_metrics = evaluate(model, val_loader, criterion, device)
print(f"Best checkpoint  val_loss={val_metrics['loss']:.4f}  macro_f1={val_metrics['macro_f1']:.4f}\n")

# collect all val predictions for the confusion matrix
model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels_list.extend(labels.tolist())

print(classification_report_str(all_labels_list, all_preds, CLASSES))


In [ ]:
# ── Task 1 Step 5: Training curves + confusion matrix ─────────────────────────
from src.evaluation.plots import plot_history, plot_confusion_matrix

fig = plot_history(history, OUT_DIR / "plots" / "task1_history.png")
plt.show(); plt.close(fig)

fig = plot_confusion_matrix(all_labels_list, all_preds, CLASSES, OUT_DIR / "plots" / "task1_confusion.png")
plt.show(); plt.close(fig)


In [ ]:
# ── Task 1 Step 6: Generate + validate submission CSV ─────────────────────────
from src.evaluation.submission import generate_submission, validate_submission

SUB_PATH = OUT_DIR / "results" / "submission_task1.csv"
generate_submission(model, test_loader, CLASSES, SUB_PATH, device)
validate_submission(SUB_PATH, expected_rows=900)
print(f"Submission saved to {SUB_PATH}")


> **Finding:** _TODO — fill in after running_